# Day 11 Tutorial — Incremental Load & MERGE Patterns

**Goal:** Watermark extracts and upsert thinking.


### Environment setup
Skip pip install on Databricks/Fabric. Locally you may need: `pip install pyspark pandas`.


In [ ]:
# %pip install pyspark==3.5.1 pandas -q


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName('AzureDE-InterviewPrep')
    .master('local[*]')
    .config('spark.sql.shuffle.partitions', '4')
    .getOrCreate()
)
spark


In [ ]:
spark.createDataFrame(
    [(1, 'Alice', 'Pune', '2024-01-01'), (2, 'Bob', 'Delhi', '2024-01-01')],
    ['id', 'name', 'city', 'last_updated'],
).createOrReplaceTempView('dim_customer_tgt')
spark.createDataFrame(
    [(2, 'Bob', 'Mumbai', '2024-01-05'), (3, 'Carol', 'London', '2024-01-05')],
    ['id', 'name', 'city', 'last_updated'],
).createOrReplaceTempView('dim_customer_src')
watermark = '2024-01-01'
spark.sql(
    "SELECT * FROM dim_customer_src WHERE last_updated > '{0}'".format(watermark)
).show()


## Upsert pattern without Delta


In [ ]:
src = spark.table('dim_customer_src')
tgt = spark.table('dim_customer_tgt')
unchanged = tgt.join(src, 'id', 'left_anti')
upserted = unchanged.unionByName(src)
upserted.orderBy('id').show()
print('In Azure SQL / Delta interviews, describe MERGE for this upsert.')


## Interview patterns
- Full vs incremental
- Idempotent reruns
- MERGE key pitfalls (duplicates in source)
